
# Late Delivery Prediction — Machine Learning

## Project Overview

This project uses the **Brazilian Olist e-commerce dataset** and **PostgreSQL** to predict late deliveries.

We explored two prediction approaches:

1. **At Purchase:** Standard and Balanced Logistic Regression with Low/Medium/High risk classification.
2. **At Carrier Handoff:** Logistic Regression and Histogram Gradient Boosting using 14 engineered features.

The purchase-time models had limited predictive performance, motivating the shift to carrier-handoff prediction.

The final model uses **Histogram Gradient Boosting** with a classification threshold of **0.15**, selected using validation F1-score.

### Objectives

- Extract and prepare order data using PostgreSQL.
- Engineer features available at prediction time.
- Compare Logistic Regression and Gradient Boosting.
- Evaluate precision, recall, F1-score, and confusion matrices.
- Analyze feature importance and potential business applications.

### Prediction Target

- **0 — Not Late:** Delivered on or before the estimated date.
- **1 — Late:** Delivered after the estimated date.

Orders without known delivery outcomes are excluded.

**Final Prediction Point:** Carrier handoff, using 14 features after excluding `approval_delay_days` due to potential data leakage.



## 1. Data Extraction

Load historical order data from the PostgreSQL `retail_bi` database using Pandas and SQLAlchemy.

We'll begin with order-level information and our binary prediction target (`is_late`). Additional features will be introduced during feature engineering.

In [685]:
# Install the PostgreSQL driver into the notebook's Python environment, psycopg2 handles PostgreSQL communication
# %pip install psycopg2-binary

import os
import pandas as pd
# SQLAlchemy manages database connections
from sqlalchemy import create_engine, text
from getpass import getpass
import sklearn

# Prompt for the database password without displaying it in the notebook
db_password = getpass("PostgreSQL password: ")

# Create a connection to our local PostgreSQL database
engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={
        "host": "localhost",
        "port": 5432,
        "dbname": "retail_bi",
        "user": "postgres",
        "password": db_password
    }
)

# Execute a simple SQL query to verify the database connection
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Connection successful:", result.scalar())

Connection successful: 1


In [686]:
# SQL query to retrieve historical orders and create our prediction target
query = """
SELECT
    order_id,
    customer_id,
    order_purchase_timestamp,
    order_estimated_delivery_date,
    delivery_status,

    -- Binary classification target: 1 = Late, 0 = Not Late
    CASE
        WHEN delivery_status = 'Late' THEN 1
        ELSE 0
    END AS is_late

FROM orders

-- Exclude orders with missing delivery outcomes
WHERE delivery_status IN ('Early', 'On Time', 'Late')
"""

# Execute the SQL query and load the results into a Pandas DataFrame
df = pd.read_sql(query, engine)

# Preview the first five rows
print(df.head())

# Display the number of rows and columns in our dataset
print("Dataset shape:", df.shape)

# Count how many orders belong to each target class
print("\nTarget distribution:")
print(df["is_late"].value_counts())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_purchase_timestamp order_estimated_delivery_date delivery_status  \
0      2017-10-02 10:56:33                    2017-10-18           Early   
1      2018-07-24 20:41:37                    2018-08-13           Early   
2      2018-08-08 08:38:49                    2018-09-04           Early   
3      2017-11-18 19:28:06                    2017-12-15           Early   
4      2018-02-13 21:18:39                    2018-02-26           Early   

   is_late  
0        0  
1        0  
2        0  
3        0  
4        0  
Dataset shape: (96476, 6)


In [687]:
# Count missing values in each column
print("Missing values:")
print(df.isna().sum())

# Check whether any order_id appears more than once
print("\nDuplicate orders:", df["order_id"].duplicated().sum())

# Inspect the data types of our columns
print("\nColumn data types:")
print(df.dtypes)


Missing values:
order_id                         0
customer_id                      0
order_purchase_timestamp         0
order_estimated_delivery_date    0
delivery_status                  0
is_late                          0
dtype: int64

Duplicate orders: 0

Column data types:
order_id                                    str
customer_id                                 str
order_purchase_timestamp         datetime64[us]
order_estimated_delivery_date    datetime64[us]
delivery_status                             str
is_late                                   int64
dtype: object


## 2. Feature Engineering

Create predictive features using information available at the time of purchase.

### Purchase Timing & Delivery Estimates
- Extract purchase month, day of week, and hour.
- Calculate the estimated delivery window in days.

### Customer Location
- Retrieve customer state information from PostgreSQL.
- Merge customer data with our order-level dataset using `customer_id`.
- Check for missing values after the merge.

Customer location may help identify regional differences in delivery reliability.

In [688]:
# Extract the month when the customer placed the order (1–12)
df["purchase_month"] = df["order_purchase_timestamp"].dt.month

# Extract the day of the week (Monday = 0, Sunday = 6)
df["purchase_day_of_week"] = df["order_purchase_timestamp"].dt.dayofweek

# Extract the hour when the order was placed (0–23)
df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

# Calculate the number of days between purchase and estimated delivery
df["estimated_delivery_days"] = (
    df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400 # converts seconds into days.

# Preview our new features alongside the prediction target
df[
    [
        "purchase_month",
        "purchase_day_of_week",
        "purchase_hour",
        "estimated_delivery_days",
        "is_late"
    ]
].head()


,purchase_month,purchase_day_of_week,purchase_hour,estimated_delivery_days,is_late
0,10,0,10,15.544063,0
1,7,1,20,19.137766,0
2,8,2,8,26.639711,0
3,11,5,19,26.188819,0
4,2,1,21,12.112049,0


In [689]:
# Retrieve customer IDs and their corresponding states from PostgreSQL
customers_query = """
SELECT
    customer_id,
    customer_state
FROM customers
"""

# Load the customer data into a separate DataFrame
customers_df = pd.read_sql(customers_query, engine)

# Merge customer states into our ML dataset using customer_id
# how="left" keeps every order from our original dataset
# validate="many_to_one" checks that each customer_id matches at most one customer record
df = df.merge(
    customers_df,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Check whether any orders are missing customer state information
print("Missing customer states:", df["customer_state"].isna().sum())

# Preview our new feature
df[["customer_id", "customer_state", "is_late"]].head()


Missing customer states: 0


,customer_id,customer_state,is_late
0,9ef432eb6251297304e76186b10a928d,SP,0
1,b0830fb4747a6c6d20dea0b8c802d7ef,BA,0
2,41ce2a54c0b03bf3443c3d931a367089,GO,0
3,f88197465ea7920adcdbec7375364d82,RN,0
4,8ab97904e6daea8866dbdbc4fb7aad2c,SP,0


### Order-Level Features

Aggregate data from `fact_order_items` to create two additional predictive features:

- **`total_order_value`** — Total merchandise value of each order.
- **`total_items`** — Number of items purchased per order.

Since `fact_order_items` contains multiple rows per order, we use SQL aggregation (`GROUP BY order_id`) before merging to preserve **one row per order** in our ML dataset.

These features may help identify relationships between order size, purchase value, and delivery delays.

In [690]:
# Aggregate order items into one row per order
order_items_query = """
SELECT
    order_id,
    SUM(price) AS total_order_value,
    COUNT(*) AS total_items
FROM fact_order_items
GROUP BY order_id
"""

# Load aggregated order information from PostgreSQL
order_items_df = pd.read_sql(order_items_query, engine)

# Merge order-level features into our ML dataset
# validate="one_to_one" ensures neither DataFrame has duplicate order IDs
df = df.merge(
    order_items_df,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Check whether any orders are missing item information
print("Missing order values:", df["total_order_value"].isna().sum())
print("Missing item counts:", df["total_items"].isna().sum())

# Confirm the number of rows hasn't changed
print("Dataset shape:", df.shape)

# Preview the new features
df[["order_id", "total_order_value", "total_items", "is_late"]].head()

# Verify that every row represents a unique order
print("Duplicate orders:", df["order_id"].duplicated().sum())



Missing order values: 0
Missing item counts: 0
Dataset shape: (96476, 13)
Duplicate orders: 0


### Seller-Level Features

Investigate seller information to identify potential features related to shipping origin and delivery performance.

- Count the number of unique sellers per order using `COUNT(DISTINCT seller_id)`.
- Determine how frequently orders contain products from multiple sellers.
- Evaluate how seller location can be incorporated while preserving **one row per order**.

Seller location may help capture geographic factors affecting delivery times and delays.

In [691]:
# Count the number of unique sellers associated with each order
seller_query = """
SELECT
    order_id,
    COUNT(DISTINCT seller_id) AS total_sellers
FROM fact_order_items
GROUP BY order_id
"""

# Load seller counts from PostgreSQL
seller_counts_df = pd.read_sql(seller_query, engine)

# Count how many orders have 1, 2, 3, etc. sellers
print("Number of sellers per order:")
print(seller_counts_df["total_sellers"].value_counts().sort_index())

Number of sellers per order:
total_sellers
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64


#### Number of Sellers per Order

Approximately **98.7% of orders contain products from a single seller**, while 1.3% involve multiple sellers.

- Merge `total_sellers` into the ML dataset using `order_id`.
- Preserve **one row per order** to prevent duplicate training records.
- Validate the merge by checking for missing values and duplicate orders.

The number of sellers may help capture differences in order fulfillment complexity and delivery risk.

In [692]:
# Merge the number of unique sellers into our ML dataset
df = df.merge(
    seller_counts_df,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Verify the merge didn't introduce missing values or duplicate orders
print("Missing seller counts:", df["total_sellers"].isna().sum())
print("Duplicate orders:", df["order_id"].duplicated().sum())

# Preview our new feature
df[["order_id", "total_sellers", "is_late"]].head()

Missing seller counts: 0
Duplicate orders: 0


,order_id,total_sellers,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,1,0
1,53cdb2fc8bc7dce0b6741e2150273451,1,0
2,47770eb9100c2d0c44946d9cf07ec65d,1,0
3,949d5b44dbf5de918fe9c16f97b45f8a,1,0
4,ad21c59c0840e6cb83a9ceb5573f8159,1,0


In [693]:
# Retrieve seller location information aggregated by order
# ARRAY_AGG(DISTINCT s.seller_state) collects unique seller states into an array for each order.
seller_location_query = """
SELECT
    f.order_id,
    COUNT(DISTINCT s.seller_state) AS total_seller_states,
    ARRAY_AGG(DISTINCT s.seller_state) AS seller_states
FROM fact_order_items f
JOIN sellers s
    ON f.seller_id = s.seller_id
GROUP BY f.order_id
"""

# Execute the query and load the results into a DataFrame
seller_location_df = pd.read_sql(seller_location_query, engine)

# Merge seller locations into our order-level ML dataset
df = df.merge(
    seller_location_df,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Check for missing values and duplicate orders
print("Missing seller locations:", df["seller_states"].isna().sum())
print("Duplicate orders:", df["order_id"].duplicated().sum())

# Preview seller locations alongside customer states
print(df[["customer_state", "seller_states", "total_seller_states"]].head())

Missing seller locations: 0
Duplicate orders: 0
  customer_state seller_states  total_seller_states
0             SP          [SP]                    1
1             BA          [SP]                    1
2             GO          [SP]                    1
3             RN          [MG]                    1
4             SP          [SP]                    1


In [694]:
# Check whether every seller state matches the customer's state
# all() returns True only if every comparison is True
df["same_state_shipping"] = df.apply(
    # function lambda takes a row, returns int value of true(1) or false(0)
    lambda row: int(
        # Returns true if all the seller states matches customer state (obvs won't if theres more than 1 seller state)
        all(
            
            state == row["customer_state"] 
            # for loop to check every state in seller_state since that is an array
            for state in row["seller_states"]
        )
    ),
    axis=1 # axis=1 - apply the function to each row, which is why we can use row as a varible in our lambda function
) 

# Count orders in each shipping category
print(df["same_state_shipping"].value_counts())

# Preview our new feature
df[["customer_state", "seller_states", "same_state_shipping", "is_late"]].head()


same_state_shipping
0    61899
1    34577
Name: count, dtype: int64


,customer_state,seller_states,same_state_shipping,is_late
0,SP,[SP],1,0
1,BA,[SP],0,0
2,GO,[SP],0,0
3,RN,[MG],0,0
4,SP,[SP],1,0


In [695]:
# Compare late delivery rates for same-state vs different-state shipping
df.groupby("same_state_shipping")["is_late"].agg(["count", "sum", "mean"])

,count,sum,mean
same_state_shipping,,,
0,61899,4972,0.080324
1,34577,1563,0.045203


#### Same-State Shipping

Created `same_state_shipping` to identify whether all sellers are located in the same state as the customer.

- `1`: All sellers are in the customer's state.
- `0`: At least one seller is in a different state.

**Findings:**
- Different-state orders: 61,899 orders, with an **8.03%** late delivery rate.
- Same-state orders: 34,577 orders, with a **4.52%** late delivery rate.
- Different-state orders were approximately **1.78 times more likely** to arrive late.

This suggests that seller-customer location differences may be a useful predictor of late deliveries.

## 3. Data Preparation

Prepare the dataset for machine learning by selecting relevant features and separating the target variable.

- **Features (`X`):** Information available at purchase time that may help predict late deliveries.
- **Target (`y`):** `is_late`, where `1` indicates a late delivery and `0` indicates an early or on-time delivery.

Exclude order identifiers, actual delivery outcomes, and other columns that could cause data leakage.

In [696]:
# Select the features we'll use to predict late deliveries
feature_columns = [
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "estimated_delivery_days",
    "customer_state",
    "total_order_value",
    "total_items",
    "total_sellers",
    "total_seller_states",
    "same_state_shipping"
]

# X contains the input features
X = df[feature_columns].copy() # .copy() creates separate copies of the selected data.

# y contains the outcome we want to predict
y = df["is_late"].copy()

# Check the dimensions
print("Features:", X.shape)
print("Target:", y.shape)

print(X.head())

Features: (96476, 10)
Target: (96476,)
   purchase_month  purchase_day_of_week  purchase_hour  \
0              10                     0             10   
1               7                     1             20   
2               8                     2              8   
3              11                     5             19   
4               2                     1             21   

   estimated_delivery_days customer_state  total_order_value  total_items  \
0                15.544063             SP              29.99            1   
1                19.137766             BA             118.70            1   
2                26.639711             GO             159.90            1   
3                26.188819             RN              45.00            1   
4                12.112049             SP              19.90            1   

   total_sellers  total_seller_states  same_state_shipping  
0              1                    1                    1  
1              1           

### Initial Experiment: 80/20

Split the dataset chronologically to simulate predicting future orders using historical data.

- **Training set (80%):** Used to train the machine learning model.
- **Testing set (20%):** Used to evaluate performance on unseen orders.

A chronological split helps prevent future orders from being used to predict earlier orders.

In [697]:
# Sort orders chronologically by their purchase timestamp
df_sorted = df.sort_values("order_purchase_timestamp")

# Calculate where to split the dataset (First 80% training, Last 20% testing)
split_index = int(len(df_sorted) * 0.8)

# Select the training and testing features
X_train = df_sorted[feature_columns].iloc[:split_index].copy()
X_test = df_sorted[feature_columns].iloc[split_index:].copy()

# Select the corresponding target values
y_train = df_sorted["is_late"].iloc[:split_index].copy()
y_test = df_sorted["is_late"].iloc[split_index:].copy()

# Check the sizes and late delivery rates
print("Training:", X_train.shape, y_train.shape)
print("Testing:", X_test.shape, y_test.shape)

print("Training late rate:", y_train.mean())
print("Testing late rate:", y_test.mean())

# Check the purchase date ranges in each dataset
print("Training period:")
print(df_sorted["order_purchase_timestamp"].iloc[:split_index].agg(["min", "max"]))

print("\nTesting period:")
print(df_sorted["order_purchase_timestamp"].iloc[split_index:].agg(["min", "max"]))

Training: (77180, 10) (77180,)
Testing: (19296, 10) (19296,)
Training late rate: 0.07593936252915263
Testing late rate: 0.03492951907131012
Training period:
min   2016-09-15 12:16:38
max   2018-05-26 16:54:05
Name: order_purchase_timestamp, dtype: datetime64[us]

Testing period:
min   2018-05-26 17:57:44
max   2018-08-29 15:00:37
Name: order_purchase_timestamp, dtype: datetime64[us]


In [698]:
# Convert customer states into numerical columns
X_train_encoded = pd.get_dummies( # pd.get_dummies() converts categorical values into numerical columns containing 0s and 1s.
    X_train, # The DataFrame we're converting.
    columns=["customer_state"], #Only encode this column.
    dtype=int # Use integers (0 and 1) instead of booleans (False and True).
)

X_test_encoded = pd.get_dummies(
    X_test,
    columns=["customer_state"],
    dtype=int
)

# Ensure testing has exactly the same feature columns as training, in case some state wasn't one-hot encoded
X_test_encoded = X_test_encoded.reindex( #.reindex() Rearranges, adds, or removes columns to match a specified set.
    columns=X_train_encoded.columns, # Use the training dataset's columns as the reference
    fill_value=0 #If a column is missing, create it and fill it with zeros.
)

print("Training:", X_train_encoded.shape)
print("Testing:", X_test_encoded.shape)

print(X_train_encoded.head())

Training: (77180, 36)
Testing: (19296, 36)
       purchase_month  purchase_day_of_week  purchase_hour  \
29811               9                     3             12   
90509              10                     0              9   
27599              10                     0             16   
89876              10                     0             21   
95058              10                     0             21   

       estimated_delivery_days  total_order_value  total_items  total_sellers  \
29811                18.488449             134.97            3              1   
90509                23.593866              29.90            1              1   
27599                34.293866              21.90            1              1   
89876                52.123831              21.50            1              1   
95058                56.115556              36.49            1              1   

       total_seller_states  same_state_shipping  customer_state_AC  ...  \
29811                 

### Feature Scaling — StandardScaler

Standardized numerical features using `StandardScaler` to ensure features with different numerical ranges are comparable during Logistic Regression training. Without scaling, features with larger numerical values can affect how efficiently Logistic Regression learns.

#### StandardScaler Formula

$$
z = \frac{x - \mu}{\sigma}
$$

Where:
- $z$ = Standardized value
- $x$ = Original feature value
- $\mu$ = Mean of the feature in the training dataset
- $\sigma$ = Standard deviation of the feature in the training dataset

After scaling, each numerical feature in the training dataset has approximately:
- Mean = 0
- Standard deviation = 1

**Implementation:**
- `fit_transform()` calculates the mean and standard deviation from training data and scales it.
- `transform()` applies those same training statistics to testing data.
- Using training statistics for both datasets prevents data leakage.
- Binary and one-hot encoded features remain as 0s and 1s.

In [699]:
from sklearn.preprocessing import StandardScaler

# Select numerical features that need scaling, leaving out the ones we one-hot encoded
numeric_features = [
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "estimated_delivery_days",
    "total_order_value",
    "total_items",
    "total_sellers",
    "total_seller_states"
]

# Create the scaler
scaler = StandardScaler()

# Learn the scaling parameters from training data
# fit_transform() calculates the mean and standard deviation from the training data, then scales it.
X_train_encoded[numeric_features] = scaler.fit_transform(X_train_encoded[numeric_features])

# Apply the same scaling parameters to testing data
X_test_encoded[numeric_features] = scaler.transform(
    X_test_encoded[numeric_features]
)

# Preview the standardized features
X_train_encoded[numeric_features].head()

,purchase_month,purchase_day_of_week,purchase_hour,estimated_delivery_days,total_order_value,total_items,total_sellers,total_seller_states
29811,0.901398,0.119700,-0.517978,-0.724945,-0.008169,3.403476,-0.108491,-0.068968
90509,1.183406,-1.407824,-1.078828,-0.091574,-0.518138,-0.264167,-0.108491,-0.068968
27599,1.183406,-1.407824,0.229821,1.235852,-0.556967,-0.264167,-0.108491,-0.068968
89876,1.183406,-1.407824,1.164571,3.447811,-0.558909,-0.264167,-0.108491,-0.068968
95058,1.183406,-1.407824,1.164571,3.943018,-0.486153,-0.264167,-0.108491,-0.068968



## 4. Model Training — Logistic Regression

Logistic Regression serves as our baseline model for predicting late deliveries (`1`) versus on-time deliveries (`0`).

### How Logistic Regression Works

**1. Calculate a weighted score:**

$$
z = b + w_1x_1 + w_2x_2 + \cdots + w_nx_n
$$

Where `b` is the intercept, `w` represents learned feature weights, and `x` represents feature values.

**2. Convert the score into a probability:**

$$
P(y=1 \mid X) = \frac{1}{1+e^{-z}}
$$

The sigmoid function produces a probability between 0 and 1.

**3. Classify the order:**

- Probability ≥ 0.5 → Late (`1`)
- Probability < 0.5 → Not Late (`0`)

The classification threshold can be adjusted to balance precision and recall.

### Model Training

- `LogisticRegression(max_iter=1000)` initializes the model.
- `fit()` learns feature weights from training data.
- `predict()` generates classifications.
- `predict_proba()` returns predicted probabilities.

### Model Evaluation

Because late deliveries are relatively uncommon, accuracy alone can be misleading.

We evaluate performance using:

- **Precision:** Percentage of predicted late orders that were actually late.
- **Recall:** Percentage of actual late orders correctly identified.
- **F1-score:** Harmonic mean of precision and recall.
- **Confusion Matrix:** Summary of correct and incorrect predictions.

### Confusion Matrix

| Actual / Predicted | Not Late (0) | Late (1) |
|---|---|---|
| **Not Late (0)** | True Negative (TN) | False Positive (FP) |
| **Late (1)** | False Negative (FN) | True Positive (TP) |

These metrics help assess the model's ability to detect late deliveries while minimizing false alerts.


In [700]:
from sklearn.linear_model import LogisticRegression

# Create a Logistic Regression model
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model using historical orders and their delivery outcomes
model.fit(X_train_encoded, y_train)

# Predict whether orders in the testing dataset will be late
y_pred = model.predict(X_test_encoded)

# Check how many orders the model predicts will be late
print(pd.Series(y_pred).value_counts())

0    19293
1        3
Name: count, dtype: int64


In [701]:
from sklearn.metrics import classification_report, confusion_matrix

# Compare the model's predictions against actual delivery outcomes
print(classification_report(y_test, y_pred, zero_division=0, digits=4)) # zero_division=0 → Returns 0 when a metric would otherwise be undefined.

# Show the number of correct and incorrect classifications
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0     0.9651    0.9999    0.9822     18622
           1     0.3333    0.0015    0.0030       674

    accuracy                         0.9650     19296
   macro avg     0.6492    0.5007    0.4926     19296
weighted avg     0.9430    0.9650    0.9480     19296

Confusion Matrix:
[[18620     2]
 [  673     1]]



### Baseline Logistic Regression — Results

**Accuracy:** 96.5%

| Metric | Late Deliveries |
|---|---:|
| Precision | 33.3% |
| Recall | 0.15% |
| F1-score | 0.003 |

**Confusion Matrix:**
- True Positives: 1
- True Negatives: 18,620
- False Positives: 2
- False Negatives: 673

### Key Findings

The dataset is **imbalanced**, with only 7.6% late deliveries in training. Consequently, the model predicts nearly all orders as on-time.

Despite 96.5% accuracy, it detected only **1 of 674 late deliveries**, making it ineffective for identifying delays.

**Next Step:** Apply `class_weight="balanced"` to penalize misclassification of late deliveries more heavily and improve recall.


In [702]:
balanced_model = LogisticRegression(
    max_iter=1000, # Allows the model up to 1,000 iterations to optimize its weights and converge during training.
    class_weight="balanced",
    random_state=42
)

# Train the balanced model
balanced_model.fit(X_train_encoded, y_train)

# Predict and evaluate
y_pred_balanced = balanced_model.predict(X_test_encoded)

print(classification_report(
    y_test, y_pred_balanced,
    zero_division=0, digits=4
))

print(confusion_matrix(y_test, y_pred_balanced))

              precision    recall  f1-score   support

           0     0.9856    0.5750    0.7263     18622
           1     0.0614    0.7685    0.1138       674

    accuracy                         0.5818     19296
   macro avg     0.5235    0.6718    0.4200     19296
weighted avg     0.9534    0.5818    0.7049     19296

[[10708  7914]
 [  156   518]]


In [703]:
# Get predicted probabilities for late deliveries (class 1)
y_prob_balanced = balanced_model.predict_proba(X_test_encoded)[:, 1]

# Try a higher probability threshold
threshold = 0.7

# Predict late only when probability reaches the threshold
y_pred_threshold = (y_prob_balanced >= threshold).astype(int)

print(classification_report(
    y_test, y_pred_threshold,
    zero_division=0,
    digits=4
))

print(confusion_matrix(y_test, y_pred_threshold))

              precision    recall  f1-score   support

           0     0.9699    0.8938    0.9303     18622
           1     0.0740    0.2344    0.1125       674

    accuracy                         0.8708     19296
   macro avg     0.5220    0.5641    0.5214     19296
weighted avg     0.9386    0.8708    0.9018     19296

[[16645  1977]
 [  516   158]]



### Balanced Logistic Regression — Results

Used `class_weight="balanced"` to give late deliveries more importance during training.

| Metric | Threshold 0.5 | Threshold 0.7 |
|---|---:|---:|
| Precision (Late) | 6.14% | 7.40% |
| Recall (Late) | 76.85% | 23.44% |
| F1-score (Late) | 0.1138 | 0.1125 |
| Accuracy | 58.18% | 87.08% |
| False Positives | 7,914 | 1,977 |
| True Positives | 518 | 158 |

**Findings:**
- `class_weight="balanced"` significantly improved recall compared with the baseline model (0.15%).
- Increasing the threshold to `0.7` reduced false alarms but missed more late deliveries.
- Neither threshold provides a strong balance between precision and recall.

### Next Steps

1. **Chronological Validation Split:** Divide the data into training (60%), validation (20%), and testing (20%) sets.

In [704]:
# Sort orders chronologically
df_sorted = df.sort_values("order_purchase_timestamp")

# Calculate split boundaries
train_end = int(len(df_sorted) * 0.60)
val_end = int(len(df_sorted) * 0.80)

# Split features
X_train = df_sorted[feature_columns].iloc[:train_end].copy()
X_val = df_sorted[feature_columns].iloc[train_end:val_end].copy()
X_test = df_sorted[feature_columns].iloc[val_end:].copy()

# Split target variable
y_train = df_sorted["is_late"].iloc[:train_end].copy()
y_val = df_sorted["is_late"].iloc[train_end:val_end].copy()
y_test = df_sorted["is_late"].iloc[val_end:].copy()

# Verify dataset sizes and late-delivery rates
for name, X, y in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Testing", X_test, y_test)
]:
    print(
        f"{name}: {X.shape} | "
        f"Late delivery rate: {y.mean():.2%}"
    )

Training: (57885, 10) | Late delivery rate: 6.81%
Validation: (19295, 10) | Late delivery rate: 9.94%
Testing: (19296, 10) | Late delivery rate: 3.49%


In [705]:
# One-hot encode customer_state
X_train_encoded = pd.get_dummies(
    X_train, columns=["customer_state"], dtype=int
)

X_val_encoded = pd.get_dummies(
    X_val, columns=["customer_state"], dtype=int
)

X_test_encoded = pd.get_dummies(
    X_test, columns=["customer_state"], dtype=int
)

# Ensure validation and testing have the same columns as training
X_val_encoded = X_val_encoded.reindex(
    columns=X_train_encoded.columns, fill_value=0
)

X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns, fill_value=0
)

# Verify shapes
print("Training:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)
print("Testing:", X_test_encoded.shape)

Training: (57885, 36)
Validation: (19295, 36)
Testing: (19296, 36)


In [706]:
from sklearn.preprocessing import StandardScaler

# Numerical features to scale
numeric_features = [
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "estimated_delivery_days",
    "total_order_value",
    "total_items",
    "total_sellers",
    "total_seller_states"
]

# Initialize scaler
scaler = StandardScaler()

# Fit scaler ONLY on training data
X_train_encoded[numeric_features] = scaler.fit_transform(
    X_train_encoded[numeric_features]
)

# Apply training scaling to validation and testing
X_val_encoded[numeric_features] = scaler.transform(
    X_val_encoded[numeric_features]
)

X_test_encoded[numeric_features] = scaler.transform(
    X_test_encoded[numeric_features]
)

# Verify scaled values
print(X_train_encoded[numeric_features].head())

       purchase_month  purchase_day_of_week  purchase_hour  \
29811        0.666752              0.112864      -0.516590   
90509        0.924337             -1.419173      -1.073505   
27599        0.924337             -1.419173       0.225963   
89876        0.924337             -1.419173       1.154155   
95058        0.924337             -1.419173       1.154155   

       estimated_delivery_days  total_order_value  total_items  total_sellers  \
29811                -0.812531          -0.001475     3.446189      -0.102694   
90509                -0.173122          -0.511445    -0.262356      -0.102694   
27599                 1.166959          -0.550274    -0.262356      -0.102694   
89876                 3.400006          -0.552216    -0.262356      -0.102694   
95058                 3.899934          -0.479460    -0.262356      -0.102694   

       total_seller_states  
29811            -0.066436  
90509            -0.066436  
27599            -0.066436  
89876            -0.0664

In [707]:
from sklearn.linear_model import LogisticRegression

# Model 1: Standard Logistic Regression
standard_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Model 2: Balanced Logistic Regression
balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train both models on the same dataset
standard_model.fit(X_train_encoded, y_train)
balanced_model.fit(X_train_encoded, y_train)

print("Both models trained successfully!")

Both models trained successfully!


In [708]:
from sklearn.metrics import classification_report, confusion_matrix

# Generate validation predictions
standard_val_pred = standard_model.predict(X_val_encoded)
balanced_val_pred = balanced_model.predict(X_val_encoded)

# Evaluate Standard Logistic Regression
print("STANDARD LOGISTIC REGRESSION")
print(classification_report(
    y_val, standard_val_pred,
    digits=4, zero_division=0
))
print(confusion_matrix(y_val, standard_val_pred))

# Evaluate Balanced Logistic Regression
print("\nBALANCED LOGISTIC REGRESSION")
print(classification_report(
    y_val, balanced_val_pred,
    digits=4, zero_division=0
))
print(confusion_matrix(y_val, balanced_val_pred))

STANDARD LOGISTIC REGRESSION
              precision    recall  f1-score   support

           0     0.9006    1.0000    0.9477     17378
           1     0.0000    0.0000    0.0000      1917

    accuracy                         0.9006     19295
   macro avg     0.4503    0.5000    0.4739     19295
weighted avg     0.8112    0.9006    0.8536     19295

[[17378     0]
 [ 1917     0]]

BALANCED LOGISTIC REGRESSION
              precision    recall  f1-score   support

           0     0.9441    0.6527    0.7718     17378
           1     0.1710    0.6495    0.2707      1917

    accuracy                         0.6523     19295
   macro avg     0.5575    0.6511    0.5212     19295
weighted avg     0.8673    0.6523    0.7220     19295

[[11342  6036]
 [  672  1245]]



## Logistic Regression — Model Evaluation

### Dataset Split

Used a **60/20/20 chronological split** for training, validation, and testing.

### Validation Results

| Metric | Standard LR | Balanced LR |
|---|---:|---:|
| Accuracy | 90.06% | 65.23% |
| Late Precision | 0.00% | 17.10% |
| Late Recall | 0.00% | 64.95% |
| Late F1-score | 0.0000 | 0.2707 |

**Key Findings:**
- Standard Logistic Regression predicted all orders as on-time.
- Balanced Logistic Regression identified **1,245 of 1,917 late deliveries**, but generated **6,036 false positives**.
- Balanced Logistic Regression improved recall at the cost of accuracy and precision.

### Three-Level Delivery Risk Classification

We used Balanced Logistic Regression scores to categorize delivery risk.

| Risk Level | Model Score | Action |
|---|---|---|
| Low | Below 0.30 | Normal processing |
| Medium | 0.30–0.69 | Monitor |
| High | 0.70 or higher | Investigate |

**Objective:** Compare actual late-delivery rates across risk groups to assess their usefulness.

**Note:** Thresholds are experimental, and model scores are not necessarily calibrated probabilities.


In [709]:
import numpy as np
"""predict_proba() returns two scores for every order:
- Column 0: Score for the order arriving on time.
- Column 1: Score for the order arriving late."""
test_risk_scores = balanced_model.predict_proba(X_test_encoded)[:, 1] # Select every row, but only the 2nd column (late-delivery score)

# Assign risk categories using the same thresholds, works like if/elif/else statements
test_risk_levels = np.select(
    [test_risk_scores < 0.30, test_risk_scores < 0.70],
    ["Low Risk","Medium Risk"],
    default="High Risk"
)

# Create testing analysis DataFrame
test_risk_analysis = pd.DataFrame({
    "risk_score": test_risk_scores, # the risk scores the model estimated
    "risk_level": test_risk_levels, # category assigned using the thresholds
    "is_late": y_test # late (1) or not late (0)
})

# Compare assigned risk categories with actual delivery outcomes
test_risk_summary = (test_risk_analysis.groupby("risk_level").agg(
    orders_predicted_at_risk_level=("is_late", "count"),
    actually_late_orders=("is_late", "sum"),
    actual_late_rate=("is_late", "mean") # contains only 0s and 1s, mean calculates the actual late rate.
    )
    .reindex(["Low Risk", "Medium Risk", "High Risk"])
)

test_risk_summary[["actual_late_rate"]] *= 100

print(test_risk_summary.round(2))

             orders_predicted_at_risk_level  actually_late_orders  \
risk_level                                                          
Low Risk                               2398                     8   
Medium Risk                           15063                   532   
High Risk                              1835                   134   

             actual_late_rate  
risk_level                     
Low Risk                 0.33  
Medium Risk              3.53  
High Risk                7.30  


In [710]:
# Get predicted risk scores from the balanced model
risk_scores = balanced_model.predict_proba(X_val_encoded)[:, 1]

# Assign risk levels based on the scores
risk_levels = np.select(
    [risk_scores < 0.30, risk_scores < 0.70],
    ["Low Risk", "Medium Risk"],
    default="High Risk"
)

# Create a DataFrame to evaluate the risk categories
risk_analysis = pd.DataFrame({
    "risk_score": risk_scores,
    "risk_level": risk_levels,
    "is_late": y_val
})

# Calculate actual late-delivery rates for each category
risk_summary = (
    risk_analysis.groupby("risk_level")
    .agg(
        total_orders=("is_late", "count"),
        late_orders=("is_late", "sum"),
        actual_late_rate=("is_late", "mean")
    )
    .reindex(["Low Risk", "Medium Risk", "High Risk"])
)

risk_summary["actual_late_rate"] *= 100

print(risk_summary.round(2))

             total_orders  late_orders  actual_late_rate
risk_level                                              
Low Risk             1267           15              1.18
Medium Risk         16489         1443              8.75
High Risk            1539          459             29.82



### Initial Approach: Purchase-Time Prediction

We trained **Standard and Balanced Logistic Regression** models using purchase-time data.

- **Standard LR:** Failed to detect most late deliveries.
- **Balanced LR:** Improved recall but generated many false positives.
- **Risk Classification:** Categorized orders using thresholds of 0.30 and 0.70.

#### Risk Classification Results

| Risk Level | Validation Late Rate | Test Late Rate |
|---|---:|---:|
| Low | 1.18% | 0.33% |
| Medium | 8.75% | 3.53% |
| High | 29.82% | 7.30% |

**Finding:** Only **134 of 1,835 high-risk test orders** were late, capturing just **19.9% of all late deliveries**. Purchase-time predictions were insufficiently reliable.

### Revised Approach: Carrier-Handoff Prediction

We shifted prediction to **carrier handoff**, incorporating shipment timing, geography, order details, and product characteristics.

#### Feature Engineering

Initially developed **15 features** across five categories:

| Category | Features |
|---|---|
| Shipment Timing | `seller_processing_days`, `remaining_delivery_days`, `shipping_delay_ratio`, `seller_shipping_delay_days`, `shipping_deadline_days` |
| Geography | `shipping_distance_km`, `same_state_shipping` |
| Order Details | `item_count`, `total_price`, `total_freight` |
| Product Details | `total_weight_g`, `total_volume_cm3` |
| Purchase Timing | `purchase_month`, `purchase_day_of_week`, `approval_delay_days` |

**Note:** Removed `approval_delay_days` after identifying **1,185 approval timestamps occurring after carrier handoff**, preventing potential data leakage. **Final model: 14 features.**

**Objective:** Compare Logistic Regression and Gradient Boosting to improve late-delivery detection while minimizing false alerts.


In [711]:
# SQL query to retrieve historical shipped orders and create our prediction target
query = """
SELECT
    order_id,
    customer_id,
    order_purchase_timestamp,
    order_delivered_carrier_date,
    order_estimated_delivery_date,
    delivery_status,

    -- Binary classification target: 1 = Late, 0 = Not Late
    CASE
        WHEN delivery_status = 'Late' THEN 1
        ELSE 0
    END AS is_late

FROM orders

-- Include only orders with known shipment and delivery outcomes
WHERE delivery_status IN ('Early', 'On Time', 'Late')
    AND order_delivered_carrier_date IS NOT NULL
"""

# Execute the SQL query and load the results into a Pandas DataFrame
df = pd.read_sql(query, engine)

# Preview the first five rows
print(df.head())

# Display the number of rows and columns
print("Dataset shape:", df.shape)

# Count the target classes
print("\nTarget distribution:")
print(df["is_late"].value_counts())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_purchase_timestamp order_delivered_carrier_date  \
0      2017-10-02 10:56:33          2017-10-04 19:55:00   
1      2018-07-24 20:41:37          2018-07-26 14:31:00   
2      2018-08-08 08:38:49          2018-08-08 13:50:00   
3      2017-11-18 19:28:06          2017-11-22 13:39:59   
4      2018-02-13 21:18:39          2018-02-14 19:46:34   

  order_estimated_delivery_date delivery_status  is_late  
0                    2017-10-18           Early        0  
1                    2018-08-13           Early        0  
2                    2018-09-

In [712]:
# Convert timestamps to datetime
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)

df["order_delivered_carrier_date"] = pd.to_datetime(
    df["order_delivered_carrier_date"]
)

# Calculate seller processing time in days
df["seller_processing_days"] = (
    df["order_delivered_carrier_date"] -
    df["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Examine the results
print(df["seller_processing_days"].describe())

count    96475.000000
mean         3.227742
std          3.598096
min       -171.212419
25%          1.127049
50%          2.201655
75%          4.067679
max        125.775521
Name: seller_processing_days, dtype: float64


In [713]:
# Count orders with negative processing times
negative_orders = df[df["seller_processing_days"] < 0]

print("Negative processing times:", len(negative_orders))

# Examine the affected orders
negative_orders[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "seller_processing_days"
    ]
].head(10)

print("Total negative orders:", len(negative_orders))

print("\nNegative processing time statistics:")
print(negative_orders["seller_processing_days"].describe())

print("\nOrders with processing time below -1 day:")
print((df["seller_processing_days"] < -1).sum())

Negative processing times: 165
Total negative orders: 165

Negative processing time statistics:
count    165.000000
mean      -1.091271
std       13.328698
min     -171.212419
25%       -0.042153
50%       -0.024213
75%       -0.009757
max       -0.000278
Name: seller_processing_days, dtype: float64

Orders with processing time below -1 day:
2


In [714]:
# Remove orders with invalid processing times
df = df[df["seller_processing_days"] >= 0].copy()

# Verify the cleaned dataset
print("Remaining orders:", len(df))
print("Negative processing times:",
      (df["seller_processing_days"] < 0).sum())

Remaining orders: 96310
Negative processing times: 0


In [715]:
# Convert estimated delivery date to datetime
df["order_estimated_delivery_date"] = pd.to_datetime(
    df["order_estimated_delivery_date"]
)

# Calculate remaining days until the delivery deadline
df["remaining_delivery_days"] = (
    df["order_estimated_delivery_date"] -
    df["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

# Examine the results
print(df["remaining_delivery_days"].describe())

# Count orders shipped after their estimated delivery deadline
print(
    "\nOrders shipped after deadline:",
    (df["remaining_delivery_days"] < 0).sum()
)

count    96310.000000
mean        20.505253
std          8.741390
min        -99.656019
25%         14.697457
50%         20.272373
75%         25.559708
max        148.101632
Name: remaining_delivery_days, dtype: float64

Orders shipped after deadline: 465


In [716]:
# Calculate the original delivery window in days
df["estimated_delivery_days"] = (
    df["order_estimated_delivery_date"] -
    df["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Calculate the proportion of the delivery window used before shipment
df["shipping_delay_ratio"] = (
    df["seller_processing_days"] /
    df["estimated_delivery_days"]
)

# Check for invalid delivery windows
print("Invalid delivery windows:",
      (df["estimated_delivery_days"] <= 0).sum())

# Examine the ratio
print(df["shipping_delay_ratio"].describe())

Invalid delivery windows: 0
count    96310.000000
mean         0.148550
std          0.163461
min          0.000013
25%          0.053369
50%          0.103192
75%          0.186513
max          7.680827
Name: shipping_delay_ratio, dtype: float64


In [717]:
# Sort orders chronologically by shipment date
df = df.sort_values(
    "order_delivered_carrier_date"
).reset_index(drop=True)

# Confirm the shipment date range
print("Earliest shipment:", df["order_delivered_carrier_date"].min())
print("Latest shipment:", df["order_delivered_carrier_date"].max())

# Verify the dataset size
print("Dataset shape:", df.shape)

Earliest shipment: 2016-10-08 10:34:01
Latest shipment: 2018-09-11 19:48:28
Dataset shape: (96310, 11)


In [718]:
query = """
SELECT
    COUNT(*) AS total_items,
    COUNT(shipping_limit_date) AS items_with_shipping_limit,
    COUNT(DISTINCT order_id) AS unique_orders
FROM order_items
"""

pd.read_sql(query, engine)

,total_items,items_with_shipping_limit,unique_orders
0,112650,112650,98666


In [719]:
query = """
SELECT
    order_id,
    MAX(shipping_limit_date) AS shipping_limit_date
FROM order_items
GROUP BY order_id
"""

shipping_deadlines = pd.read_sql(query, engine)

shipping_deadlines["shipping_limit_date"] = pd.to_datetime(
    shipping_deadlines["shipping_limit_date"]
)

df = df.merge(shipping_deadlines, on="order_id", how="left")

print("Missing deadlines:", df["shipping_limit_date"].isna().sum())
print("Dataset shape:", df.shape)

df["seller_shipping_delay_days"] = (
    df["order_delivered_carrier_date"] -
    df["shipping_limit_date"]
).dt.total_seconds() / 86400

df.groupby("is_late")["seller_shipping_delay_days"].describe().round(2)

Missing deadlines: 0
Dataset shape: (96310, 12)


,count,mean,std,min,25%,50%,75%,max
is_late,,,,,,,,
0,89778.0,-3.55,4.78,-1046.86,-5.26,-3.32,-1.72,41.32
1,6532.0,-0.45,6.93,-32.92,-4.31,-1.98,0.30,116.76


In [720]:
df.loc[
    df["seller_shipping_delay_days"].abs() > 100,
    [
        "order_id",
        "order_delivered_carrier_date",
        "shipping_limit_date",
        "seller_shipping_delay_days"
    ]
]

df = df[df["seller_shipping_delay_days"].abs() <= 100].copy()

print("Dataset shape:", df.shape)

Dataset shape: (96308, 13)


In [721]:
query = """
SELECT
    oi.order_id,
    MAX(CASE
        WHEN c.customer_state = s.seller_state THEN 1
        ELSE 0
    END) AS same_state_shipping
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
JOIN sellers s ON oi.seller_id = s.seller_id
GROUP BY oi.order_id
"""

shipping_locations = pd.read_sql(query, engine)

df = df.merge(shipping_locations, on="order_id", how="left")

print(df.groupby("is_late")["same_state_shipping"].mean())
print("Missing values:", df["same_state_shipping"].isna().sum())

is_late
0    0.369961
1    0.239320
Name: same_state_shipping, dtype: float64
Missing values: 0


In [722]:
query = """
SELECT
    COUNT(DISTINCT geolocation_zip_code_prefix) AS unique_zip_codes,
    COUNT(*) AS total_records,
    COUNT(geolocation_lat) AS available_latitudes,
    COUNT(geolocation_lng) AS available_longitudes
FROM geolocation
"""

pd.read_sql(query, engine)

,unique_zip_codes,total_records,available_latitudes,available_longitudes
0,19015,19015,19015,19015


In [723]:
import numpy as np

query = """
SELECT
    oi.order_id,
    MAX(
        6371 * 2 * ASIN(
            SQRT(
                POWER(SIN(RADIANS(cg.geolocation_lat - sg.geolocation_lat) / 2), 2)
                + COS(RADIANS(sg.geolocation_lat))
                * COS(RADIANS(cg.geolocation_lat))
                * POWER(SIN(RADIANS(cg.geolocation_lng - sg.geolocation_lng) / 2), 2)
            )
        )
    ) AS shipping_distance_km
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
JOIN sellers s ON oi.seller_id = s.seller_id
LEFT JOIN geolocation cg
    ON c.customer_zip_code_prefix = cg.geolocation_zip_code_prefix
LEFT JOIN geolocation sg
    ON s.seller_zip_code_prefix = sg.geolocation_zip_code_prefix
GROUP BY oi.order_id
"""

shipping_distances = pd.read_sql(query, engine)

df = df.merge(shipping_distances, on="order_id", how="left")

print("Missing distances:", df["shipping_distance_km"].isna().sum())
print(df.groupby("is_late")["shipping_distance_km"].mean().round(2))

Missing distances: 476
is_late
0    589.81
1    768.35
Name: shipping_distance_km, dtype: float64


In [724]:
# Keep the existing chronological split
train_end = int(len(df) * 0.60)

median_distance = df.iloc[:train_end]["shipping_distance_km"].median()

df["shipping_distance_km"] = df["shipping_distance_km"].fillna(
    median_distance
)

print("Missing distances:", df["shipping_distance_km"].isna().sum())
print("Median used:", round(median_distance, 2), "km")

Missing distances: 0
Median used: 454.36 km


In [725]:
query = """
SELECT
    oi.order_id,
    COUNT(*) AS item_count,
    SUM(oi.price) AS total_price,
    SUM(oi.freight_value) AS total_freight,
    SUM(p.product_weight_g) AS total_weight_g,
    SUM(
        p.product_length_cm *
        p.product_height_cm *
        p.product_width_cm
    ) AS total_volume_cm3
FROM order_items oi
LEFT JOIN products p
    ON oi.product_id = p.product_id
GROUP BY oi.order_id
"""

order_features = pd.read_sql(query, engine)

df = df.merge(order_features, on="order_id", how="left")

print(df[order_features.columns].isna().sum())
print("Dataset shape:", df.shape)

order_id             0
item_count           0
total_price          0
total_freight        0
total_weight_g      16
total_volume_cm3    16
dtype: int64
Dataset shape: (96308, 20)


In [726]:
# Purchase timing
df["purchase_month"] = df["order_purchase_timestamp"].dt.month
df["purchase_day_of_week"] = df["order_purchase_timestamp"].dt.dayofweek

# Seller shipping deadline window
df["shipping_deadline_days"] = (
    df["shipping_limit_date"] -
    df["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Retrieve payment approval timestamps
query = """
SELECT order_id, order_approved_at
FROM orders
"""

approval_dates = pd.read_sql(query, engine)

approval_dates["order_approved_at"] = pd.to_datetime(
    approval_dates["order_approved_at"]
)

df = df.merge(approval_dates, on="order_id", how="left")

# Time taken to approve the order
df["approval_delay_days"] = (
    df["order_approved_at"] -
    df["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Check whether any approvals occurred after carrier handoff
print(
    "Approvals after carrier handoff:",
    (
        df["order_approved_at"] >
        df["order_delivered_carrier_date"]
    ).sum()
)

print(df[[
    "purchase_month",
    "purchase_day_of_week",
    "shipping_deadline_days",
    "approval_delay_days"
]].isna().sum())

print("Dataset shape:", df.shape)

Approvals after carrier handoff: 1185
purchase_month             0
purchase_day_of_week       0
shipping_deadline_days     0
approval_delay_days       14
dtype: int64
Dataset shape: (96308, 25)



### Feature Removal — Preventing Data Leakage

We found that **1,185 orders** had approval timestamps occurring after carrier handoff.

Since predictions are made at carrier handoff, `approval_delay_days` could introduce **data leakage** by using information unavailable at prediction time.

Therefore, we removed `approval_delay_days` from the model, reducing the final feature count from **15 to 14**.


In [733]:
missing_columns = [
    "total_weight_g",
    "total_volume_cm3"
]

# Use training data only to calculate medians
train_end = int(len(df) * 0.60)

for column in missing_columns:
    median_value = df.iloc[:train_end][column].median()
    df[column] = df[column].fillna(median_value)

# Check remaining missing values
print("Missing values:")
print(df[missing_columns].isna().sum())

print("\nShipping deadline statistics:")
print(
    df[["shipping_deadline_days"]]
    .describe()
    .round(2)
)

Missing values:
total_weight_g      0
total_volume_cm3    0
dtype: int64

Shipping deadline statistics:
       shipping_deadline_days
count                96308.00
mean                     6.57
std                      3.22
min                      2.00
25%                      5.01
50%                      6.01
75%                      7.14
max                     97.05



### Feature Engineering: Shipment-Time Prediction

We initially engineered **15 features** covering shipment timing, geography, order characteristics, and seasonality. After removing one feature due to potential data leakage, **14 features were retained**.

#### Final Model Features

| Feature | Description |
|---|---|
| `seller_processing_days` | Days from purchase to carrier handoff. |
| `remaining_delivery_days` | Days remaining until estimated delivery. |
| `shipping_delay_ratio` | Fraction of delivery window used before shipment. |
| `seller_shipping_delay_days` | Days between shipping deadline and carrier handoff. |
| `same_state_shipping` | Whether at least one seller shares the customer's state. |
| `shipping_distance_km` | Maximum estimated customer-to-seller distance. |
| `item_count` | Number of ordered items. |
| `total_price` | Total product value. |
| `total_freight` | Total shipping cost. |
| `total_weight_g` | Combined product weight. |
| `total_volume_cm3` | Combined product volume. |
| `purchase_month` | Purchase month. |
| `purchase_day_of_week` | Purchase weekday. |
| `shipping_deadline_days` | Days from purchase to seller shipping deadline. |

**Feature Removal:** Excluded `approval_delay_days` because **1,185 approval timestamps occurred after carrier handoff**, creating potential data leakage.

#### Data Cleaning and Preparation

- Started with **96,475 orders** with known delivery outcomes.
- Removed **165 invalid processing times** and **2 extreme shipping-delay outliers**.
- Retained **96,308 orders** for modeling.
- Imputed missing distances (476), weights (16), and volumes (16) using training-data medians.
- Applied a **chronological 60/20/20 train/validation/test split**.
- Standardized features for Logistic Regression; Gradient Boosting used unscaled data.


In [728]:
feature_columns = [
    "seller_processing_days",
    "remaining_delivery_days",
    "shipping_delay_ratio",
    "seller_shipping_delay_days",
    "same_state_shipping",
    "shipping_distance_km",
    "item_count",
    "total_price",
    "total_freight",
    "total_weight_g",
    "total_volume_cm3",
    "purchase_month",
    "purchase_day_of_week",
    "shipping_deadline_days",
]

X = df[feature_columns]
y = df["is_late"]

train_end = int(len(df) * 0.60)
val_end = int(len(df) * 0.80)

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]

y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:val_end]
y_test = y.iloc[val_end:]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Training:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Testing:", X_test_scaled.shape)

Training: (57784, 14)
Validation: (19262, 14)
Testing: (19262, 14)


In [729]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

balanced_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

balanced_model.fit(X_train_scaled, y_train)

balanced_predictions = balanced_model.predict(X_val_scaled)

print("Balanced Logistic Regression:")
print(classification_report(
    y_val,
    balanced_predictions,
    zero_division=0
))

from sklearn.metrics import precision_score, recall_score, f1_score

balanced_probabilities = balanced_model.predict_proba(X_val_scaled)[:, 1]

for threshold in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
    predictions = (balanced_probabilities >= threshold).astype(int)

    print(
        f"{threshold:.2f} | "
        f"Precision: {precision_score(y_val, predictions, zero_division=0):.2%} | "
        f"Recall: {recall_score(y_val, predictions, zero_division=0):.2%} | "
        f"F1: {f1_score(y_val, predictions, zero_division=0):.2%}"
    )

Balanced Logistic Regression:
              precision    recall  f1-score   support

           0       0.96      0.63      0.76     17294
           1       0.19      0.77      0.31      1968

    accuracy                           0.65     19262
   macro avg       0.58      0.70      0.53     19262
weighted avg       0.88      0.65      0.72     19262

0.50 | Precision: 19.18% | Recall: 76.68% | F1: 30.68%
0.55 | Precision: 21.73% | Recall: 66.21% | F1: 32.72%
0.60 | Precision: 24.68% | Recall: 54.73% | F1: 34.02%
0.65 | Precision: 28.71% | Recall: 43.19% | F1: 34.49%
0.70 | Precision: 33.74% | Recall: 33.33% | F1: 33.54%
0.75 | Precision: 40.50% | Recall: 24.59% | F1: 30.60%
0.80 | Precision: 50.42% | Recall: 18.14% | F1: 26.68%


In [730]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report

gb_model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_predictions = gb_model.predict(X_val)

print("Gradient Boosting:")
print(classification_report(
    y_val,
    gb_predictions,
    zero_division=0
))

from sklearn.metrics import precision_score, recall_score, f1_score

gb_probabilities = gb_model.predict_proba(X_val)[:, 1]

for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    predictions = (gb_probabilities >= threshold).astype(int)

    print(
        f"{threshold:.2f} | "
        f"Precision: {precision_score(y_val, predictions, zero_division=0):.2%} | "
        f"Recall: {recall_score(y_val, predictions, zero_division=0):.2%} | "
        f"F1: {f1_score(y_val, predictions, zero_division=0):.2%}"
    )

Gradient Boosting:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     17294
           1       0.81      0.07      0.12      1968

    accuracy                           0.90     19262
   macro avg       0.86      0.53      0.53     19262
weighted avg       0.89      0.90      0.86     19262

0.10 | Precision: 27.78% | Recall: 56.55% | F1: 37.26%
0.15 | Precision: 34.06% | Recall: 42.33% | F1: 37.74%
0.20 | Precision: 38.24% | Recall: 29.83% | F1: 33.51%
0.25 | Precision: 45.41% | Recall: 21.34% | F1: 29.04%
0.30 | Precision: 53.57% | Recall: 14.48% | F1: 22.80%
0.35 | Precision: 64.94% | Recall: 10.16% | F1: 17.57%
0.40 | Precision: 72.81% | Recall: 8.03% | F1: 14.46%
0.45 | Precision: 78.45% | Recall: 7.22% | F1: 13.22%
0.50 | Precision: 81.13% | Recall: 6.55% | F1: 12.13%


In [731]:
from sklearn.metrics import classification_report, confusion_matrix

final_model = gb_model
final_threshold = 0.15

test_probabilities = final_model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= final_threshold).astype(int)

print("Final Test Results:")
print(classification_report(
    y_test,
    test_predictions,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

Final Test Results:
              precision    recall  f1-score   support

           0       0.98      0.97      0.97     18558
           1       0.38      0.54      0.44       704

    accuracy                           0.95     19262
   macro avg       0.68      0.75      0.71     19262
weighted avg       0.96      0.95      0.95     19262


Confusion Matrix:
[[17931   627]
 [  324   380]]


In [734]:
from sklearn.inspection import permutation_importance
import pandas as pd

# Calculate feature importance using the final 14-feature model
importance = permutation_importance(
    final_model,
    X_val,
    y_val,
    scoring="average_precision",
    n_repeats=5,
    random_state=42
)

feature_importance = pd.DataFrame({
    "Feature": X_val.columns,
    "Importance": importance.importances_mean
}).sort_values("Importance", ascending=False)

print(feature_importance.round(4))

                       Feature  Importance
1      remaining_delivery_days      0.1192
5         shipping_distance_km      0.0722
4          same_state_shipping      0.0527
11              purchase_month      0.0397
2         shipping_delay_ratio      0.0318
8                total_freight      0.0217
6                   item_count      0.0048
0       seller_processing_days      0.0034
13      shipping_deadline_days      0.0030
3   seller_shipping_delay_days      0.0026
10            total_volume_cm3      0.0009
12        purchase_day_of_week      0.0002
9               total_weight_g     -0.0007
7                  total_price     -0.0014



## Final Model Evaluation

### Model Configuration

- **Algorithm:** HistGradientBoostingClassifier
- **Features:** 14 shipment, geographic, order, product, and date-related features
- **Prediction Point:** Carrier handoff
- **Data Split:** Chronological (60% train, 20% validation, 20% test)
- **Classification Threshold:** 0.15

### Test Set Performance

| Metric | Result |
|---|---:|
| Accuracy | 95.06% |
| Precision (Late) | 37.74% |
| Recall (Late) | 53.98% |
| F1-score (Late) | 44.42% |

### Confusion Matrix

| | Predicted On-Time | Predicted Late |
|---|---:|---:|
| Actual On-Time | 17,931 | 627 |
| Actual Late | 324 | 380 |

### Key Findings

- Correctly identified **380 of 704 late deliveries (54%)**.
- Approximately **38% of flagged orders** were actually late.
- Generated **627 false positives** and missed **324 late deliveries**.
- Gradient Boosting achieved a better precision–recall balance than Logistic Regression.

### Most Important Features

Permutation importance was calculated using the final 14-feature Gradient Boosting model on validation data.

| Rank | Feature | Importance |
|---|---|---:|
| 1 | Remaining delivery days | 0.1192 |
| 2 | Shipping distance | 0.0722 |
| 3 | Same-state shipping | 0.0527 |
| 4 | Purchase month | 0.0397 |
| 5 | Shipping delay ratio | 0.0318 |
| 6 | Total freight cost | 0.0217 |

**Key Finding:** Remaining delivery time and geographic shipping factors were the strongest predictors of late deliveries.

### Business Applications

The model could help retailers prioritize high-risk shipments, monitor potential delays, and proactively notify customers.

### Limitations

- Missed approximately **46% of late deliveries**.
- Approximately **62% of alerts were false positives**.
- Late-delivery rates differed between validation (10.2%) and testing (3.7%).
- Further validation is required before production deployment.

### Conclusion

The final **14-feature Gradient Boosting model** achieved a **44.42% F1-score** on the test set, outperforming Logistic Regression on validation data.

While not production-ready, it demonstrates the potential of predictive analytics for improving shipment monitoring and logistics decisions.


In [732]:
# Close all idle database connections managed by the engine
engine.dispose()